In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

# 1. Load Data
selected_features_df = pd.read_csv('../../data/03_primary/selected_features.csv')
churn_data = pd.read_csv('../../data/03_primary/churn_features.csv')

# 2. Prepare Features (X) and Target (y)
# Extract the list of features we want to keep
selected_features = selected_features_df['feature'].tolist()
target_col = 'customer_churn'

# Filter the dataframe to only include selected features + target
valid_features = [f for f in selected_features if f in churn_data.columns]
X = churn_data[valid_features].copy()
y = churn_data[target_col]

# Handle Missing Values 
X = X.fillna(0)

# 3. Train/Test Split (80/20)
# stratify=y ensures we keep the same ratio of churners in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Handle Class Imbalance
# Heavily penalizes the model if it misses an actual churner
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# 5. Train Gradient Boosting Model
print("Training Gradient Boosting Classifier...")
model = HistGradientBoostingClassifier(
    max_iter=150,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)
model.fit(X_train, y_train, sample_weight=sample_weights)

# 6. Evaluate Model
y_pred = model.predict(X_test)

print("\n" + "="*40)
print("MODEL EVALUATION")
print("="*40)
print("Classification Report:\n", classification_report(y_test, y_pred))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 7. Feature Importance Calculation
print("\nCalculating Feature Importances...")
result = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': result.importances_mean
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Important Features:")
print(importance_df.head(10))

Training Gradient Boosting Classifier...

MODEL EVALUATION
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.92      0.94      3329
           1       0.77      0.88      0.82       977

    accuracy                           0.91      4306
   macro avg       0.87      0.90      0.88      4306
weighted avg       0.92      0.91      0.91      4306

Confusion Matrix:
 [[3071  258]
 [ 117  860]]

Calculating Feature Importances...

Top 10 Important Features:
                   Feature  Importance
0                total_van    0.295727
1                  avg_van    0.274222
3          total_bob_value    0.045982
12    company_size_encoded    0.008175
2             num_machines    0.007571
10           num_contracts    0.005667
7   avg_agreement_duration    0.003251
8   typical_creation_month    0.003065
15        avg_repair_ratio    0.001765
4       total_annual_value    0.000697


In [2]:
y.head(10)

0    1
1    0
2    0
3    0
4    0
5    1
6    0
7    0
8    0
9    0
Name: customer_churn, dtype: int64